In [3]:
import zipfile

with zipfile.ZipFile("novatech_dataset_part1.zip", "r") as zip_ref:
    zip_ref.extractall("data")

In [4]:
import os

print(os.listdir("data"))

['company_profile.txt', 'warranty_policy.txt', 'faq.txt', 'shipping_policy.txt', 'refund_policy.txt', 'contact_information.txt', 'products.txt', 'promotions.txt', 'troubleshooting.txt', 'returns.txt']


In [5]:
with open("data/company_profile.txt", "r", encoding="utf-8") as f:
    print(f.read())

NovaTech Electronics Company Profile

NovaTech Electronics is an online retailer of laptops, smartphones, tablets, wearables and accessories. Mission: provide reliable technology at affordable prices.

Business Hours:
Mon-Fri 9:00 AM-6:00 PM
Sat 10:00 AM-4:00 PM
Sun Closed

Support:
Email: support@novatech.example
Phone: +1-800-555-0100

Head Office:
100 Innovation Drive, San Francisco, CA



In [6]:
import os

documents = []

for filename in os.listdir("data"):
    if filename.endswith(".txt"):
        with open(os.path.join("data", filename), "r", encoding="utf-8") as f:
            documents.append({
                "filename": filename,
                "content": f.read()
            })

print(f"Loaded {len(documents)} documents")

Loaded 10 documents


In [7]:


print(documents[0]["filename"])
print(documents[0]["content"][:300])

company_profile.txt
NovaTech Electronics Company Profile

NovaTech Electronics is an online retailer of laptops, smartphones, tablets, wearables and accessories. Mission: provide reliable technology at affordable prices.

Business Hours:
Mon-Fri 9:00 AM-6:00 PM
Sat 10:00 AM-4:00 PM
Sun Closed

Support:
Email: support@n


In [11]:
import os

documents = []

for filename in os.listdir("data"):
    if filename.endswith(".txt"):
        with open(os.path.join("data", filename), "r", encoding="utf-8") as f:
            documents.append({
                "source": filename,
                "text": f.read()
            })

print(f"Loaded {len(documents)} documents")

Loaded 10 documents


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = []

for doc in documents:
    split_text = splitter.split_text(doc["text"])

    for chunk in split_text:
        chunks.append({
            "source": doc["source"],
            "text": chunk
        })

print("Total Chunks:", len(chunks))

In [ ]:
print(chunks[0])

{'source': 'company_profile.txt', 'text': 'NovaTech Electronics Company Profile\n\nNovaTech Electronics is an online retailer of laptops, smartphones, tablets, wearables and accessories. Mission: provide reliable technology at affordable prices.\n\nBusiness Hours:\nMon-Fri 9:00 AM-6:00 PM\nSat 10:00 AM-4:00 PM\nSun Closed'}


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print("Embeddings Shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings Shape: (12, 384)


In [ ]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="customer_support"
)

In [ ]:
for i, chunk in enumerate(chunks):
    collection.add(
        ids=[str(i)],
        documents=[chunk["text"]],
        metadatas=[{"source": chunk["source"]}],
        embeddings=[embeddings[i].tolist()]
    )

print("Stored all chunks in ChromaDB!")

Stored all chunks in ChromaDB!


In [ ]:
query = "What is your refund policy?"

In [ ]:
query_embedding = embedding_model.encode(query).tolist()

In [ ]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

In [ ]:
for i, doc in enumerate(results["documents"][0]):
    print(f"\nResult {i+1}")
    print(doc)


Result 1
Refund Policy

Products may be returned within 30 days of delivery.
Items must be unused, in original packaging, with accessories.
Refunds are issued within 5-7 business days after inspection.
Shipping charges are refunded only for defective or incorrect items.

Result 2
Frequently Asked Questions

Q: How do I track my order?
A: Use the tracking link emailed after shipment.

Q: Can I cancel my order?
A: Yes, before it has shipped.

Q: Which payment methods are accepted?
A: Visa, MasterCard, PayPal, Apple Pay and Google Pay.

Result 3
Q: How do I contact support?
A: Email support@novatech.example or call +1-800-555-0100.


In [ ]:
import google.generativeai as genai

genai.configure(api_key="YOUR_API_KEY")

model = genai.GenerativeModel("gemini-2.5-flash")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


### Securely Storing and Accessing API Keys in Colab

To avoid hardcoding your API key directly in your notebook, which is a security risk, you can use Colab's built-in **Secrets Manager**.

1.  **Open Secrets Manager**: Click on the "🔑" (key) icon in the left sidebar of your Colab notebook.
2.  **Add a new secret**: Click "+ New secret" and add your API key. It's recommended to name it `GOOGLE_API_KEY` for consistency with common practices.
3.  **Enable notebook access**: Make sure the toggle next to your secret is enabled for your notebook to access it.

Now, you can access your API key in your code like this:

After updating the API key in Secrets Manager and running the above cell, the previous `BadRequest` error should be resolved, and you can proceed with generating content using the `model`.

In [ ]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

In [ ]:
context = "\n\n".join(results["documents"][0])

print(context)

Refund Policy

Products may be returned within 30 days of delivery.
Items must be unused, in original packaging, with accessories.
Refunds are issued within 5-7 business days after inspection.
Shipping charges are refunded only for defective or incorrect items.

Frequently Asked Questions

Q: How do I track my order?
A: Use the tracking link emailed after shipment.

Q: Can I cancel my order?
A: Yes, before it has shipped.

Q: Which payment methods are accepted?
A: Visa, MasterCard, PayPal, Apple Pay and Google Pay.

Q: How do I contact support?
A: Email support@novatech.example or call +1-800-555-0100.


In [ ]:
prompt = f"""
You are a professional customer service assistant for NovaTech Electronics.

Answer ONLY using the provided context.

If the answer is not available in the context, reply:

"I couldn't find that information in the company knowledge base."

Context:
{context}

Question:
{query}
"""

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="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",
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
client = OpenAI(
    api_key="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",
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": "Say hello in one sentence."
        }
    ]
)

print(response.choices[0].message.content)

Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
print(collection.count())

12


In [ ]:
query = "What is your refund policy?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

In [ ]:
context = "\n\n".join(results["documents"][0])

print(context)

Refund Policy

Products may be returned within 30 days of delivery.
Items must be unused, in original packaging, with accessories.
Refunds are issued within 5-7 business days after inspection.
Shipping charges are refunded only for defective or incorrect items.

Frequently Asked Questions

Q: How do I track my order?
A: Use the tracking link emailed after shipment.

Q: Can I cancel my order?
A: Yes, before it has shipped.

Q: Which payment methods are accepted?
A: Visa, MasterCard, PayPal, Apple Pay and Google Pay.

Q: How do I contact support?
A: Email support@novatech.example or call +1-800-555-0100.


In [ ]:
prompt = f"""
You are NovaTech Electronics' AI Customer Service Assistant.

Answer ONLY using the context below.

If the answer is not in the context, say:
"I couldn't find that information in the company knowledge base."

Context:
{context}

Question:
{query}
"""

In [ ]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful customer support assistant."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.choices[0].message.content)

Our refund policy is as follows: Products may be returned within 30 days of delivery. Items must be unused, in original packaging, with accessories. Refunds are issued within 5-7 business days after inspection. Shipping charges are refunded only for defective or incorrect items.


In [ ]:
import streamlit as st
    query = input("\nAsk your question (type 'exit' to quit): ")

    if query.lower() == "exit":
        print("Thank you for using NovaTech Customer Support!")
        break

    # Convert the question into an embedding
    query_embedding = embedding_model.encode(query).tolist()

    # Retrieve relevant chunks
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    # Build context
    context = "\n\n".join(results["documents"][0])

    # Create prompt
    prompt = f"""
You are NovaTech Electronics' AI Customer Service Assistant.

Answer ONLY using the provided context.

If the answer is not found in the context, reply:
"I couldn't find that information in the company knowledge base."

Context:
{context}

Question:
{query}
"""

    # Generate response
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful customer support assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    print("\nAnswer:")
    print(response.choices[0].message.content)

    print("\nRetrieved Sources:")
    for metadata in results["metadatas"][0]:
        print("-", metadata["source"])


Answer:
To track your order, use the tracking link that was emailed to you after shipment.

Retrieved Sources:
- returns.txt
- faq.txt
- shipping_policy.txt

Answer:
Products may be returned within 30 days of delivery. Items must be unused, in original packaging, with accessories. Refunds are issued within 5-7 business days after inspection. Shipping charges are refunded only for defective or incorrect items.

Retrieved Sources:
- refund_policy.txt
- faq.txt
- faq.txt

Answer:
According to our Shipping Policy, Standard Shipping takes 3-5 business days. Since you've mentioned it's already 3 days late, I would recommend checking your email for the tracking number that was emailed after dispatch. You can use this to track the status of your order. If you're still concerned, please allow up to 5 business days for standard shipping, or 2 business days for express shipping, before contacting us further.

Retrieved Sources:
- refund_policy.txt
- shipping_policy.txt
- returns.txt
